In [ ]:
import time
import torch
import torch.nn as nn
from torch.nn import functional as F
from datasets import load_dataset
from transformers import GPT2Tokenizer

# Hyperparameters:

# No. of training examples processed at once.
batch_size = 8
# Context length.
block_size = 256
# Max no of iterations.
max_iters = 1500
# No of iterations after which evaluation happens.
eval_interval = 300
# Step-size during gradient descent.
learning_rate = 3e-4
# Choosing a gpu or a cpu.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# No of batches to use while evaluating the validation loss. Since each batch has 8 training examples so in total 8 * 50 = 400 examples are being used to calculate the validation loss.
eval_iters = 50
# Dimension of each token in the vector space.
n_embd = 128
# No of attention heads required while implementing Multi-Head attention. Each head shall get n_embd / n_head = 128 / 2 = 32 dimensions.
n_head = 4
# No of transformer blocks
n_layer = 4
# randomly switches of 20% of the neurons at each step to prevent overfitting.
dropout = 0.2

# To make random operations reproducible.
torch.manual_seed(1337)

# loading the WikiText-2 dataset.
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
# loading the GPT2 tokenizer.
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# GPT2 tokenizer by default has no padding, padding is done to make the
# sentences of the same length. We are using the "EOS"(End of Sequence) token for padding.
tokenizer.pad_token = tokenizer.eos_token

# Tokenize function:
def tokenize_function(examples):
    return tokenizer(examples["text"])

# Tokenize the whole dataset:
tokenized_datasets = dataset.map(
    tokenize_function,
    # To tokenize a batch_size no of examples at once.
    batched=True,
    # After tokenization we remove the raw text.
    remove_columns=["text"]
)

# Converts to one long token sequence
train_ids = []
for item in tokenized_datasets["train"]["input_ids"]:
    train_ids.extend(item)

val_ids = []
for item in tokenized_datasets["validation"]["input_ids"]:
    val_ids.extend(item)

# convert to tensors
train_data = torch.tensor(train_ids, dtype=torch.long)
val_data = torch.tensor(val_ids, dtype=torch.long)

# vocabulary size
vocab_size = tokenizer.vocab_size

# decode function
decode = lambda l: tokenizer.decode(l)

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

# To evalulate how well the model is performing on training and validation data.
@torch.no_grad() # Tells PyTorch to not compute the gradients cause during evaluation we are only measuring loss and not training.
# Computes average training and validation loss.
def estimate_loss():
    out = {}
    model.eval() # Going to evaluation mode, dropout doesnt happen during evaluation mode.
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train() # Back to training mode.
    return out

# Multi-head causal linear attention
class LinearAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.num_heads = num_heads
        self.head_size = head_size #num_heads = n_head
        self.key = nn.Linear(n_embd, num_heads * head_size, bias=False)
        self.query = nn.Linear(n_embd, num_heads * head_size, bias=False)
        self.value = nn.Linear(n_embd, num_heads * head_size, bias=False)
        self.proj = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def feature_map(self, x):
        # ELU(x) = x, x > 0 and e^x - 1, x <= 0.
        return F.elu(x) + 1 # "+ 1" since ELU(x) can produce negative values   upto -1.

    def forward(self, x):

        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        # Reshape into "num_heads" number of "n_embd / num_heads" dimensional vectors.
        k = k.view(B, T, self.num_heads, self.head_size).transpose(1, 2)
        q = q.view(B, T, self.num_heads, self.head_size).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_size).transpose(1, 2)

        # Apply kernel feature map
        q = self.feature_map(q)
        k = self.feature_map(k)

        # Compute outer products k_t v_t^T
        kv = k.unsqueeze(-1) * v.unsqueeze(-2) # (B, H, T, D, D)


        # Prefix sums
        kv_cumsum = kv.cumsum(dim=2) # (B, H, T, D, D)


        k_cumsum = k.cumsum(dim=2) # (B, H, T, D)


        # Numerator
        numerator = torch.matmul(
            q.unsqueeze(-2),
            kv_cumsum
        ).squeeze(-2)
        # (B, H, T, D)

        # Denominator
        denominator = (
            (q * k_cumsum).sum(-1, keepdim=True)
            + 1e-6
        )
        # (B, H, T, 1), 1e-6 is added to prevent division by zero

        # Final attention output
        out = numerator / denominator
        # (B, H, T, D)

        # Merge heads
        out = out.transpose(1, 2).contiguous()
        out = out.view(B, T, self.num_heads * self.head_size)

        out = self.proj(out)
        out = self.dropout(out)

        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            # Expands the total dimensions from n_embd -> 4 * n_embd. (here: 128--> 512)
            nn.Linear(n_embd, 4 * n_embd),
            # ReLU(x) = max(0, x)
            nn.ReLU(),
            # Compress back to n_embd dimensions.
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = LinearAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        # Layernorms, normalizes activations to stabilize training.
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # "Residual Connection", instead of replacing the old representations completely
        # the model learns the corrections and the refinements.
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # Final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights) # Initializing with random small weights(centered at 0 and std dev = 0.02) and biases.

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # "idx" is the full sequence generated so far.
        for _ in range(max_new_tokens):
            # "idx_cond" is the last "block_size" no of tokens in idx.
            idx_cond = idx[:, -block_size:]
            # Get the predictions.
            logits, loss = self(idx_cond)
            # Focus only on the last time step.
            logits = logits[:, -1, :] # Becomes (B, C).
            # Apply softmax to get probabilities.
            probs = F.softmax(logits, dim=-1) # (B, C)
            # Sample from the distribution.
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # Append sampled index to the running sequence.
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = GPTLanguageModel().to(device)
# Print the number of parameters in the model.
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

# Create a PyTorch optimizer.
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

start_time = time.time()
last_eval_iter = 0

for iter in range(max_iters):

    # Every once in a while evaluate the loss on train and val sets.
    if (iter > 0 and iter % eval_interval == 0) or iter == max_iters - 1:

        if torch.cuda.is_available():
          torch.cuda.reset_peak_memory_stats()

        losses = estimate_loss()

        if torch.cuda.is_available():
          torch.cuda.synchronize()

        elapsed = time.time() - start_time

        iterations_done = iter - last_eval_iter
        tokens_processed = iterations_done * batch_size * block_size

        throughput = 0 if iterations_done == 0 else tokens_processed / elapsed

        # Perplexity
        train_ppl = torch.exp(losses['train'])
        val_ppl = torch.exp(losses['val'])

        print(f"\nstep {iter}:")
        print(f"train ppl: {train_ppl:.2f}")
        print(f"val ppl: {val_ppl:.2f}")
        print(f"throughput: {throughput:.2f} tokens/sec")

        if torch.cuda.is_available():
          memory = torch.cuda.max_memory_allocated() / 1024**3
          print(f"gpu memory: {memory:.2f} GB")

        start_time = time.time()
        last_eval_iter = iter

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()



# INFERENCE BENCHMARK
# Puts the model into Eval mode. Different from training mode for example dropout doesnt happen in eval mode but rather only in training mode.
model.eval()
# No of tokens to generate.
gen_tokens = 200
# Creates a random token sequence from 0 to "vocab_size". (1, block_size) is the shape of the tensor. Why do we have random context instead of zeroes? Cause attention variants matter mostly at LONG contexts.
# This random context simulates realistic inference.
context = torch.randint(
    0,
    vocab_size,
    (1, block_size),
    device=device
)
# CUDA operations are asynchronous.
if torch.cuda.is_available():
    torch.cuda.synchronize()

start = time.time()

# We dont calculate the gradients since there is no backpropagation, optimizer updates so it'd be useless to compute them.
with torch.no_grad():
    model.generate(context, max_new_tokens=gen_tokens)

# Synchronize Again to ensure that the GPU fully finishes generation before stopping the timer.
if torch.cuda.is_available():
    torch.cuda.synchronize()

elapsed = time.time() - start

print("\nInference Benchmark:")
print(f"Inference tokens/sec: {gen_tokens / elapsed:.2f}")
print(f"Latency per token: {elapsed / gen_tokens:.4f} sec")



# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=500)[0].tolist()))
#open('more.txt', 'w').write(decode(m.generate(context, max_new_tokens=10000)[0].tolist()))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/4358 [00:00<?, ? examples/s]

Map:   0%|          | 0/36718 [00:00<?, ? examples/s]

Map:   0%|          | 0/3760 [00:00<?, ? examples/s]

13.740625 M parameters

step 300:
train ppl: 921.44
val ppl: 965.00
throughput: 29949.24 tokens/sec
gpu memory: 1.76 GB

step 600:
train ppl: 590.64
val ppl: 654.14
throughput: 28445.94 tokens/sec
gpu memory: 1.76 GB

step 900:
train ppl: 467.12
val ppl: 538.83
throughput: 30176.13 tokens/sec
gpu memory: 1.76 GB

step 1200:
train ppl: 371.51
val ppl: 459.83
throughput: 29658.43 tokens/sec
gpu memory: 1.76 GB

step 1499:
train ppl: 335.19
val ppl: 438.41
throughput: 28385.20 tokens/sec
gpu memory: 1.76 GB

Inference Benchmark:
Inference tokens/sec: 222.46
Latency per token: 0.0045 sec
! and The teacher 2003 near three MPs and moves for 50 draw with mixed minist Marc gang of least 6 – sprint towns and repaired . 
 Col seeks to makeinea was added that a traditional host case of recreational regHOU Andy On a heroine . Caroline Winterman thinks " Urological Right for the production . several episodeswanzrow served as an act into the v exaggeration necessary to Resolution 100 , tying Black c

In [ ]:
import time
import torch
import torch.nn as nn
from torch.nn import functional as F
from datasets import load_dataset
from transformers import GPT2Tokenizer

# Hyperparameters:

# No. of training examples processed at once.
batch_size = 8
# Context length.
block_size = 512
# Max no of iterations.
max_iters = 1500
# No of iterations after which evaluation happens.
eval_interval = 300
# Step-size during gradient descent.
learning_rate = 3e-4
# Choosing a gpu or a cpu.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# No of batches to use while evaluating the validation loss. Since each batch has 8 training examples so in total 8 * 50 = 400 examples are being used to calculate the validation loss.
eval_iters = 50
# Dimension of each token in the vector space.
n_embd = 128
# No of attention heads required while implementing Multi-Head attention. Each head shall get n_embd / n_head = 128 / 2 = 32 dimensions.
n_head = 4
# No of transformer blocks
n_layer = 4
# randomly switches of 20% of the neurons at each step to prevent overfitting.
dropout = 0.2

# To make random operations reproducible.
torch.manual_seed(1337)

# loading the WikiText-2 dataset.
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
# loading the GPT2 tokenizer.
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# GPT2 tokenizer by default has no padding, padding is done to make the
# sentences of the same length. We are using the "EOS"(End of Sequence) token for padding.
tokenizer.pad_token = tokenizer.eos_token

# Tokenize function:
def tokenize_function(examples):
    return tokenizer(examples["text"])

# Tokenize the whole dataset:
tokenized_datasets = dataset.map(
    tokenize_function,
    # To tokenize a batch_size no of examples at once.
    batched=True,
    # After tokenization we remove the raw text.
    remove_columns=["text"]
)

# Converts to one long token sequence
train_ids = []
for item in tokenized_datasets["train"]["input_ids"]:
    train_ids.extend(item)

val_ids = []
for item in tokenized_datasets["validation"]["input_ids"]:
    val_ids.extend(item)

# convert to tensors
train_data = torch.tensor(train_ids, dtype=torch.long)
val_data = torch.tensor(val_ids, dtype=torch.long)

# vocabulary size
vocab_size = tokenizer.vocab_size

# decode function
decode = lambda l: tokenizer.decode(l)

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

# To evalulate how well the model is performing on training and validation data.
@torch.no_grad() # Tells PyTorch to not compute the gradients cause during evaluation we are only measuring loss and not training.
# Computes average training and validation loss.
def estimate_loss():
    out = {}
    model.eval() # Going to evaluation mode, dropout doesnt happen during evaluation mode.
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train() # Back to training mode.
    return out

#num_heads = n_head
class LinearAttention(nn.Module):
    """
    Multi-head causal linear attention (Performer-style)
    """

    def __init__(self, num_heads, head_size):
        super().__init__()

        self.num_heads = num_heads
        self.head_size = head_size

        self.key = nn.Linear(n_embd, num_heads * head_size, bias=False)
        self.query = nn.Linear(n_embd, num_heads * head_size, bias=False)
        self.value = nn.Linear(n_embd, num_heads * head_size, bias=False)

        self.proj = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def feature_map(self, x):
        # Performer-style positive kernel feature map
        # ELU(x) = x, x > 0 and e^x - 1, x <= 0.
        return F.elu(x) + 1 # "+ 1" since ELU(x) can produce negative values   upto -1.

    def forward(self, x):

        B, T, C = x.shape

        # Project
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        # Reshape into "num_heads" number of "n_embd / num_heads" dimensional vectors.
        k = k.view(B, T, self.num_heads, self.head_size).transpose(1, 2)
        q = q.view(B, T, self.num_heads, self.head_size).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_size).transpose(1, 2)

        # Apply kernel feature map
        q = self.feature_map(q)
        k = self.feature_map(k)

        # -----------------------------------
        # Compute outer products k_t v_t^T
        # -----------------------------------

        kv = k.unsqueeze(-1) * v.unsqueeze(-2)
        # (B, H, T, D, D)

        # -----------------------------------
        # Prefix sums
        # -----------------------------------

        kv_cumsum = kv.cumsum(dim=2)
        # (B, H, T, D, D)

        k_cumsum = k.cumsum(dim=2)
        # (B, H, T, D)

        # -----------------------------------
        # Numerator
        # -----------------------------------

        numerator = torch.matmul(
            q.unsqueeze(-2),
            kv_cumsum
        ).squeeze(-2)
        # (B, H, T, D)

        # -----------------------------------
        # Denominator
        # -----------------------------------

        denominator = (
            (q * k_cumsum).sum(-1, keepdim=True)
            + 1e-6
        )
        # (B, H, T, 1)

        # -----------------------------------
        # Final attention output
        # -----------------------------------

        out = numerator / denominator
        # (B, H, T, D)

        # Merge heads
        out = out.transpose(1, 2).contiguous()
        out = out.view(B, T, self.num_heads * self.head_size)

        out = self.proj(out)
        out = self.dropout(out)

        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            # Expands the total dimensions from n_embd -> 4 * n_embd. (here: 128--> 512)
            nn.Linear(n_embd, 4 * n_embd),
            # ReLU(x) = max(0, x)
            nn.ReLU(),
            # Compress back to n_embd dimensions.
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = LinearAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        # Layernorms, normalizes activations to stabilize training.
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # "Residual Connection", instead of replacing the old representations completely
        # the model learns the corrections and the refinements.
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # Final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights) # Initializing with random small weights(centered at 0 and std dev = 0.02) and biases.

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # "idx" is the full sequence generated so far.
        for _ in range(max_new_tokens):
            # "idx_cond" is the last "block_size" no of tokens in idx.
            idx_cond = idx[:, -block_size:]
            # Get the predictions.
            logits, loss = self(idx_cond)
            # Focus only on the last time step.
            logits = logits[:, -1, :] # Becomes (B, C).
            # Apply softmax to get probabilities.
            probs = F.softmax(logits, dim=-1) # (B, C)
            # Sample from the distribution.
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # Append sampled index to the running sequence.
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = GPTLanguageModel().to(device)
# Print the number of parameters in the model.
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

# Create a PyTorch optimizer.
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

start_time = time.time()
last_eval_iter = 0

for iter in range(max_iters):

    # Every once in a while evaluate the loss on train and val sets.
    if (iter > 0 and iter % eval_interval == 0) or iter == max_iters - 1:

        if torch.cuda.is_available():
          torch.cuda.reset_peak_memory_stats()

        losses = estimate_loss()

        if torch.cuda.is_available():
          torch.cuda.synchronize()

        elapsed = time.time() - start_time

        iterations_done = iter - last_eval_iter
        tokens_processed = iterations_done * batch_size * block_size

        throughput = 0 if iterations_done == 0 else tokens_processed / elapsed

        # Perplexity
        train_ppl = torch.exp(losses['train'])
        val_ppl = torch.exp(losses['val'])

        print(f"\nstep {iter}:")
        print(f"train ppl: {train_ppl:.2f}")
        print(f"val ppl: {val_ppl:.2f}")
        print(f"throughput: {throughput:.2f} tokens/sec")

        if torch.cuda.is_available():
          memory = torch.cuda.max_memory_allocated() / 1024**3
          print(f"gpu memory: {memory:.2f} GB")

        start_time = time.time()
        last_eval_iter = iter

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()



# INFERENCE BENCHMARK
# Puts the model into Eval mode. Different from training mode for example dropout doesnt happen in eval mode but rather only in training mode.
model.eval()
# No of tokens to generate.
gen_tokens = 200
# Creates a random token sequence from 0 to "vocab_size". (1, block_size) is the shape of the tensor. Why do we have random context instead of zeroes? Cause attention variants matter mostly at LONG contexts.
# This random context simulates realistic inference.
context = torch.randint(
    0,
    vocab_size,
    (1, block_size),
    device=device
)
# CUDA operations are asynchronous.
if torch.cuda.is_available():
    torch.cuda.synchronize()

start = time.time()

# We dont calculate the gradients since there is no backpropagation, optimizer updates so it'd be useless to compute them.
with torch.no_grad():
    model.generate(context, max_new_tokens=gen_tokens)

# Synchronize Again to ensure that the GPU fully finishes generation before stopping the timer.
if torch.cuda.is_available():
    torch.cuda.synchronize()

elapsed = time.time() - start

print("\nInference Benchmark:")
print(f"Inference tokens/sec: {gen_tokens / elapsed:.2f}")
print(f"Latency per token: {elapsed / gen_tokens:.4f} sec")



# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=500)[0].tolist()))
#open('more.txt', 'w').write(decode(m.generate(context, max_new_tokens=10000)[0].tolist()))

13.773393 M parameters

step 300:
train ppl: 852.72
val ppl: 902.36
throughput: 28256.11 tokens/sec
gpu memory: 3.30 GB

step 600:
train ppl: 519.53
val ppl: 578.21
throughput: 29148.78 tokens/sec
gpu memory: 3.30 GB

step 900:
train ppl: 402.30
val ppl: 486.57
throughput: 28642.91 tokens/sec
gpu memory: 3.30 GB

step 1200:
train ppl: 331.23
val ppl: 434.45
throughput: 28945.83 tokens/sec
gpu memory: 3.30 GB

step 1499:
train ppl: 277.13
val ppl: 412.26
throughput: 28765.16 tokens/sec
gpu memory: 3.30 GB

Inference Benchmark:
Inference tokens/sec: 200.91
Latency per token: 0.0050 sec
! of the winter , the Clark taxClr of Game , midfield return panna ( Broitt ) , re shaped characters lose to their e.Republican , along with a 5 in London , butiderBiian College , now due to only one model . In ) , and covere 's , the player is a woman . dive , Baltimore , including his stal posts , was Grutation , that a cultured in Paris musone as ensures that wore the next merged with those meaning of t

In [ ]:
import time
import torch
import torch.nn as nn
from torch.nn import functional as F
from datasets import load_dataset
from transformers import GPT2Tokenizer

# Hyperparameters:

# No. of training examples processed at once.
batch_size = 8
# Context length.
block_size = 1024
# Max no of iterations.
max_iters = 1500
# No of iterations after which evaluation happens.
eval_interval = 300
# Step-size during gradient descent.
learning_rate = 3e-4
# Choosing a gpu or a cpu.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# No of batches to use while evaluating the validation loss. Since each batch has 8 training examples so in total 8 * 50 = 400 examples are being used to calculate the validation loss.
eval_iters = 50
# Dimension of each token in the vector space.
n_embd = 128
# No of attention heads required while implementing Multi-Head attention. Each head shall get n_embd / n_head = 128 / 2 = 32 dimensions.
n_head = 4
# No of transformer blocks
n_layer = 4
# randomly switches of 20% of the neurons at each step to prevent overfitting.
dropout = 0.2

# To make random operations reproducible.
torch.manual_seed(1337)

# loading the WikiText-2 dataset.
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
# loading the GPT2 tokenizer.
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# GPT2 tokenizer by default has no padding, padding is done to make the
# sentences of the same length. We are using the "EOS"(End of Sequence) token for padding.
tokenizer.pad_token = tokenizer.eos_token

# Tokenize function:
def tokenize_function(examples):
    return tokenizer(examples["text"])

# Tokenize the whole dataset:
tokenized_datasets = dataset.map(
    tokenize_function,
    # To tokenize a batch_size no of examples at once.
    batched=True,
    # After tokenization we remove the raw text.
    remove_columns=["text"]
)

# Converts to one long token sequence
train_ids = []
for item in tokenized_datasets["train"]["input_ids"]:
    train_ids.extend(item)

val_ids = []
for item in tokenized_datasets["validation"]["input_ids"]:
    val_ids.extend(item)

# convert to tensors
train_data = torch.tensor(train_ids, dtype=torch.long)
val_data = torch.tensor(val_ids, dtype=torch.long)

# vocabulary size
vocab_size = tokenizer.vocab_size

# decode function
decode = lambda l: tokenizer.decode(l)

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

# To evalulate how well the model is performing on training and validation data.
@torch.no_grad() # Tells PyTorch to not compute the gradients cause during evaluation we are only measuring loss and not training.
# Computes average training and validation loss.
def estimate_loss():
    out = {}
    model.eval() # Going to evaluation mode, dropout doesnt happen during evaluation mode.
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train() # Back to training mode.
    return out

#num_heads = n_head
class LinearAttention(nn.Module):
    """
    Multi-head causal linear attention (Performer-style)
    """

    def __init__(self, num_heads, head_size):
        super().__init__()

        self.num_heads = num_heads
        self.head_size = head_size

        self.key = nn.Linear(n_embd, num_heads * head_size, bias=False)
        self.query = nn.Linear(n_embd, num_heads * head_size, bias=False)
        self.value = nn.Linear(n_embd, num_heads * head_size, bias=False)

        self.proj = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def feature_map(self, x):
        # Performer-style positive kernel feature map
        # ELU(x) = x, x > 0 and e^x - 1, x <= 0.
        return F.elu(x) + 1 # "+ 1" since ELU(x) can produce negative values   upto -1.

    def forward(self, x):

        B, T, C = x.shape

        # Project
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        # Reshape into "num_heads" number of "n_embd / num_heads" dimensional vectors.
        k = k.view(B, T, self.num_heads, self.head_size).transpose(1, 2)
        q = q.view(B, T, self.num_heads, self.head_size).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_size).transpose(1, 2)

        # Apply kernel feature map
        q = self.feature_map(q)
        k = self.feature_map(k)

        # -----------------------------------
        # Compute outer products k_t v_t^T
        # -----------------------------------

        kv = k.unsqueeze(-1) * v.unsqueeze(-2)
        # (B, H, T, D, D)

        # -----------------------------------
        # Prefix sums
        # -----------------------------------

        kv_cumsum = kv.cumsum(dim=2)
        # (B, H, T, D, D)

        k_cumsum = k.cumsum(dim=2)
        # (B, H, T, D)

        # -----------------------------------
        # Numerator
        # -----------------------------------

        numerator = torch.matmul(
            q.unsqueeze(-2),
            kv_cumsum
        ).squeeze(-2)
        # (B, H, T, D)

        # -----------------------------------
        # Denominator
        # -----------------------------------

        denominator = (
            (q * k_cumsum).sum(-1, keepdim=True)
            + 1e-6
        )
        # (B, H, T, 1)

        # -----------------------------------
        # Final attention output
        # -----------------------------------

        out = numerator / denominator
        # (B, H, T, D)

        # Merge heads
        out = out.transpose(1, 2).contiguous()
        out = out.view(B, T, self.num_heads * self.head_size)

        out = self.proj(out)
        out = self.dropout(out)

        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            # Expands the total dimensions from n_embd -> 4 * n_embd. (here: 128--> 512)
            nn.Linear(n_embd, 4 * n_embd),
            # ReLU(x) = max(0, x)
            nn.ReLU(),
            # Compress back to n_embd dimensions.
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = LinearAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        # Layernorms, normalizes activations to stabilize training.
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # "Residual Connection", instead of replacing the old representations completely
        # the model learns the corrections and the refinements.
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # Final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights) # Initializing with random small weights(centered at 0 and std dev = 0.02) and biases.

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # "idx" is the full sequence generated so far.
        for _ in range(max_new_tokens):
            # "idx_cond" is the last "block_size" no of tokens in idx.
            idx_cond = idx[:, -block_size:]
            # Get the predictions.
            logits, loss = self(idx_cond)
            # Focus only on the last time step.
            logits = logits[:, -1, :] # Becomes (B, C).
            # Apply softmax to get probabilities.
            probs = F.softmax(logits, dim=-1) # (B, C)
            # Sample from the distribution.
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # Append sampled index to the running sequence.
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = GPTLanguageModel().to(device)
# Print the number of parameters in the model.
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

# Create a PyTorch optimizer.
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

start_time = time.time()
last_eval_iter = 0

for iter in range(max_iters):

    # Every once in a while evaluate the loss on train and val sets.
    if (iter > 0 and iter % eval_interval == 0) or iter == max_iters - 1:

        if torch.cuda.is_available():
          torch.cuda.reset_peak_memory_stats()

        losses = estimate_loss()

        if torch.cuda.is_available():
          torch.cuda.synchronize()

        elapsed = time.time() - start_time

        iterations_done = iter - last_eval_iter
        tokens_processed = iterations_done * batch_size * block_size

        throughput = 0 if iterations_done == 0 else tokens_processed / elapsed

        # Perplexity
        train_ppl = torch.exp(losses['train'])
        val_ppl = torch.exp(losses['val'])

        print(f"\nstep {iter}:")
        print(f"train ppl: {train_ppl:.2f}")
        print(f"val ppl: {val_ppl:.2f}")
        print(f"throughput: {throughput:.2f} tokens/sec")

        if torch.cuda.is_available():
          memory = torch.cuda.max_memory_allocated() / 1024**3
          print(f"gpu memory: {memory:.2f} GB")

        start_time = time.time()
        last_eval_iter = iter

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()



# INFERENCE BENCHMARK
# Puts the model into Eval mode. Different from training mode for example dropout doesnt happen in eval mode but rather only in training mode.
model.eval()
# No of tokens to generate.
gen_tokens = 200
# Creates a random token sequence from 0 to "vocab_size". (1, block_size) is the shape of the tensor. Why do we have random context instead of zeroes? Cause attention variants matter mostly at LONG contexts.
# This random context simulates realistic inference.
context = torch.randint(
    0,
    vocab_size,
    (1, block_size),
    device=device
)
# CUDA operations are asynchronous.
if torch.cuda.is_available():
    torch.cuda.synchronize()

start = time.time()

# We dont calculate the gradients since there is no backpropagation, optimizer updates so it'd be useless to compute them.
with torch.no_grad():
    model.generate(context, max_new_tokens=gen_tokens)

# Synchronize Again to ensure that the GPU fully finishes generation before stopping the timer.
if torch.cuda.is_available():
    torch.cuda.synchronize()

elapsed = time.time() - start

print("\nInference Benchmark:")
print(f"Inference tokens/sec: {gen_tokens / elapsed:.2f}")
print(f"Latency per token: {elapsed / gen_tokens:.4f} sec")



# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=500)[0].tolist()))
#open('more.txt', 'w').write(decode(m.generate(context, max_new_tokens=10000)[0].tolist()))

13.838929 M parameters

step 300:
train ppl: 754.41
val ppl: 792.13
throughput: 28927.20 tokens/sec
gpu memory: 6.37 GB

step 600:
train ppl: 460.11
val ppl: 531.37
throughput: 28908.10 tokens/sec
gpu memory: 6.37 GB

step 900:
train ppl: 341.08
val ppl: 451.40
throughput: 28958.86 tokens/sec
gpu memory: 6.37 GB

step 1200:
train ppl: 282.71
val ppl: 398.36
throughput: 28936.50 tokens/sec
gpu memory: 6.37 GB

step 1499:
train ppl: 236.87
val ppl: 370.61
throughput: 28984.87 tokens/sec
gpu memory: 6.37 GB

Inference Benchmark:
Inference tokens/sec: 105.33
Latency per token: 0.0095 sec
!ada life , according to the greater my Roy Petan , like Meyer serving the Maraia , the search for Castle , a sparkiate Act of elements of common pieceography double flight of the Mean Buryi ( a knowledge and the ride , though the Conoles encompeconus . At Cleveland , whose evidence an principle exist in the H-@ paid the clockian habitat , Colorado Moor on a shaft total of boarding other flights were other